In [ ]:
import atexit
import hashlib
import json
import os
import threading
import time
from pathlib import Path

RUN_ID = "trackastra-graph-finetune-v4"
STARTED = time.monotonic()
FINISHED = False
TERMINAL = Path("/kaggle/working/launcher_terminal.json")


def write_terminal(status, error=None):
    training_terminal = Path("/kaggle/working/trackastra_graph_v4/training_terminal.json")
    payload = {
        "run_id": RUN_ID,
        "status": status,
        "elapsed_seconds": round(time.monotonic() - STARTED, 3),
        "declared_budget_seconds": 14400,
        "hard_stop_seconds": 14100,
        "training_terminal_exists": training_terminal.is_file(),
    }
    if error:
        payload["error"] = str(error)
    if training_terminal.is_file():
        payload["training_terminal_sha256"] = hashlib.sha256(training_terminal.read_bytes()).hexdigest()
    temporary = TERMINAL.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(TERMINAL)


def abort_terminal():
    if not FINISHED:
        write_terminal("aborted")


def budget_expired():
    write_terminal("budget_expired")
    os._exit(124)


atexit.register(abort_terminal)
TIMER = threading.Timer(14100, budget_expired)
TIMER.daemon = True
TIMER.start()
print("Trackastra fine-tune watchdog armed for 14,100 seconds.")


# Biohub-native Trackastra association fine-tuning

This is a new candidate pretrained on corrected CC0 synthetic sequence graphs, then fine-tuned on Biohub graph supervision. Public 0.927 outputs are used only as frozen detector inputs for clean validation.


In [ ]:
import importlib
import shutil
import subprocess
import sys
import zipfile


def first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)


input_root = Path("/kaggle/input")
runtime = first_existing([
    Path("/kaggle/input/datasets/indarkarhana/biohub-trackastra-graph-runtime-v1"),
    Path("/kaggle/input/biohub-trackastra-graph-runtime-v1"),
])
competition = first_existing([
    Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
    Path("/kaggle/input/biohub-cell-tracking-during-development"),
])
support = first_existing([
    Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"),
    Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1"),
])
synthetic = next(
    (
        path.parent
        for path in input_root.rglob("manifest.json")
        if (path.parent / "sequences").is_dir()
    ),
    None,
)
if runtime is None:
    runtime = next(
        (
            p
            for p in input_root.iterdir()
            if (p / "SOURCE_MANIFEST.json").is_file()
            or (p / "runtime_bundle.zip").is_file()
        ),
        None,
    )
if support is None:
    support = next((p for p in input_root.iterdir() if (p / "wheels").is_dir()), None)
if runtime is None or competition is None or support is None or synthetic is None:
    raise FileNotFoundError({
        "runtime": runtime,
        "competition": competition,
        "support": support,
        "synthetic": synthetic,
    })

runtime_bundle = runtime / "runtime_bundle.zip"
if runtime_bundle.is_file():
    materialized_runtime = Path("/kaggle/working/runtime_bundle")
    if materialized_runtime.exists():
        shutil.rmtree(materialized_runtime)
    materialized_runtime.mkdir(parents=True)
    with zipfile.ZipFile(runtime_bundle) as handle:
        handle.extractall(materialized_runtime)
    runtime = materialized_runtime
if not (runtime / "SOURCE_MANIFEST.json").is_file():
    raise FileNotFoundError(f"Runtime manifest missing after materialization: {runtime}")


def materialize_runtime_directory(name, marker):
    direct = runtime / name
    if (direct / marker).exists():
        return direct
    archive = runtime / f"{name}.zip"
    if not archive.is_file():
        raise FileNotFoundError(f"Missing runtime directory/archive: {name}")
    destination = Path("/kaggle/working/runtime_materialized") / name
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
    for candidate in (destination, destination / name):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Archive {archive} did not contain {marker}")


ctc_dir = materialize_runtime_directory("ctc", "model.pt")
trackastra_dir = materialize_runtime_directory(
    "trackastra_source", "trackastra/model/model.py"
)
validation_dir = materialize_runtime_directory(
    "validator_raw", "44b6_12dfb391.geff/zarr.json"
)

wheel_dirs = sorted({p.parent for p in support.rglob("*.whl")})
if not wheel_dirs:
    raise FileNotFoundError(f"No offline wheels found below {support}")
numpy_before = importlib.import_module("numpy").__version__
if numpy_before != "2.0.2":
    raise RuntimeError(f"Unexpected Kaggle NumPy before offline install: {numpy_before}")

# The support pack was resolved against a newer NumPy and tracksdata declares
# ``numpy>2`` even though its graph IO works with Kaggle's NumPy 2.0.2.  Letting
# pip resolve the pack either rejects 2.0.2 or replaces it with 2.4.6, breaking
# Kaggle's compiled SciPy stack.  Install only the graph-IO wheels we need and
# deliberately leave the live numerical stack untouched.
pip_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
for wheel_dir in wheel_dirs:
    pip_cmd.extend(["--find-links", str(wheel_dir)])
pip_cmd.extend([
    "bidict==0.23.1",
    "donfig==0.8.1.post1",
    "geff==1.2.0.1.1",
    "geff-spec==1.1.1",
    "ilpy==0.6.0",
    "numcodecs==0.15.1",
    "polars==1.42.0",
    "polars-runtime-32==1.42.0",
    "pyscipopt==6.2.1",
    "rustworkx==0.18.0",
    "tracksdata==0.1.0rc6.dev3+g980c2d30a",
    "zarr==3.2.1",
])
print("Installing verified offline dependencies from", support)
subprocess.run(pip_cmd, check=True)
for module in ("numpy", "scipy", "tracksdata", "zarr", "polars", "dask", "yaml"):
    importlib.import_module(module)
numpy_after = importlib.import_module("numpy").__version__
if numpy_after != numpy_before:
    raise RuntimeError(f"Offline install changed NumPy: {numpy_before} -> {numpy_after}")
from scipy.optimize import linear_sum_assignment
linear_sum_assignment([[0.0, 1.0], [1.0, 0.0]])

probe = importlib.import_module("tracksdata").graph.IndexedRXGraph.from_geff(
    validation_dir / "44b6_12dfb391.geff"
)
probe = probe[0] if isinstance(probe, tuple) else probe
if probe.node_attrs().height <= 0 or probe.edge_attrs().height <= 0:
    raise RuntimeError("Offline graph-IO probe returned an empty validation graph")
print("Offline numerical stack and complete GEFF graph probe passed.")

manifest = json.loads((runtime / "SOURCE_MANIFEST.json").read_text(encoding="utf-8"))
expected_model_hash = manifest["pretrained_model"]["model_sha256"]
actual_model_hash = hashlib.sha256((ctc_dir / "model.pt").read_bytes()).hexdigest()
if actual_model_hash != expected_model_hash:
    raise RuntimeError(f"Pretrained model hash mismatch: {actual_model_hash}")
print(json.dumps({
    "runtime": str(runtime),
    "competition": str(competition),
    "support": str(support),
    "synthetic": str(synthetic),
    "model_sha256": actual_model_hash,
}, indent=2))


In [ ]:
output_dir = Path("/kaggle/working/trackastra_graph_v4")
command = [
    sys.executable,
    str(runtime / "trainer.py"),
    "--competition-dir", str(competition),
    "--trackastra-dir", str(trackastra_dir),
    "--pretrained-dir", str(ctc_dir),
    "--validation-predictions", str(validation_dir),
    "--output-dir", str(output_dir),
    "--steps", "7000",
    "--train-per-prefix", "128",
    "--max-tokens", "512",
    "--gradient-accumulation", "4",
    "--validation-samples", "24",
    "--max-wall-seconds", "13200",
    "--validation-reserve-seconds", "1800",
    "--synthetic-root", str(synthetic),
    "--synthetic-graphs", "384",
    "--synthetic-steps", "1200",
    "--synthetic-learning-rate-multiplier", "2.0",
    "--synthetic-prefer-division-probability", "0.15",
]
print("Launching Biohub-native Trackastra fine-tuning:", " ".join(command))
try:
    subprocess.run(command, check=True)
except Exception as exc:
    write_terminal("failed", exc)
    raise

training_terminal = output_dir / "training_terminal.json"
if not training_terminal.is_file():
    raise RuntimeError("Trainer exited without terminal evidence")
result = json.loads(training_terminal.read_text(encoding="utf-8"))
print(json.dumps({
    "step": result["step"],
    "elapsed_seconds": result["elapsed_seconds"],
    "association_acceptance_passed": result["association_acceptance_passed"],
    "selection_delta_vs_base_raw": result["selection_delta_vs_base_raw"],
    "acceptance_delta_vs_base_raw": result["acceptance_delta_vs_base_raw"],
    "selected_method": result["complete_movie_selected"]["method"],
    "acceptance_summary": result["complete_movie_selected"]["acceptance_summary"],
}, indent=2))


In [ ]:
FINISHED = True
TIMER.cancel()
write_terminal("completed")
print("Experiment complete; evidence is in", Path("/kaggle/working/trackastra_graph_v4"))
